# Phase 4: Retrieval Diagnostics

This notebook collects retrieval-quality signals before generation. It does not apply thresholds, classify failures, retry retrieval, or make routing decisions.

## Environment

Replace `REPOSITORY_URL` with your repository URL before running this notebook in Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')

if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL to your GitHub repository URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unavailable')

## Initialize dense, BM25, hybrid, and reranking components

In [ ]:
from src.rag import (
    BM25Retriever,
    CrossEncoderReranker,
    FAISSRetriever,
    HybridRetriever,
    RAGConfig,
    compute_retrieval_diagnostics,
    load_documents,
)

config = RAGConfig()
documents = load_documents(Path('data/phase1_corpus.json'))
index_dir = Path('data/phase1_faiss_index')

dense_retriever = FAISSRetriever(
    embedding_model_name=config.embedding_model_name,
    device=config.device,
    batch_size=config.embedding_batch_size,
)
if (index_dir / 'documents.faiss').exists():
    dense_retriever.load(index_dir)
    print(f'Loaded FAISS index with {len(dense_retriever.documents)} documents.')
else:
    dense_retriever.build(documents)
    dense_retriever.save(index_dir)
    print(f'Built FAISS index with {len(documents)} documents.')

bm25_retriever = BM25Retriever(documents)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)
reranker = CrossEncoderReranker(device=config.device)
print('Reranker:', reranker.model_name)
print('Reranker device:', reranker.device)

## Inspect candidates and diagnostics

The diagnostic values are observations only. No value is treated as a pass/fail boundary.

In [ ]:
def print_ranked_results(label, results, score_attribute='score'):
    print(f'\n{label}')
    for rank, result in enumerate(results, start=1):
        score = getattr(result, score_attribute)
        print(f'  {rank}. {result.title} (id={result.id}, score={score:.6f})')
        print(f'     {result.text}')

In [ ]:
def inspect_query(query: str, retrieval_k: int = 5, candidate_k: int = 10):
    dense_results = dense_retriever.retrieve(query, top_k=retrieval_k)
    bm25_results = bm25_retriever.retrieve(query, top_k=retrieval_k)
    hybrid_results = hybrid_retriever.retrieve(query, top_k=candidate_k)
    reranked_results = reranker.rerank(query, hybrid_results)
    diagnostics = compute_retrieval_diagnostics(
        query=query,
        dense_results=dense_results,
        bm25_results=bm25_results,
        hybrid_results=hybrid_results,
        reranked_results=reranked_results,
    )

    print('\n' + '=' * 100)
    print('Query:', query)
    print_ranked_results('Dense candidates', dense_results)
    print_ranked_results('BM25 candidates', bm25_results)
    print_ranked_results('Hybrid candidates', hybrid_results)
    print_ranked_results('Cross-encoder ranking', reranked_results, 'reranker_score')

    print('\nFull retrieval diagnostics')
    for field, value in diagnostics.to_dict().items():
        print(f'  {field}: {value}')
    return diagnostics

## Query scenarios

The final query is intentionally outside the corpus. Retrieval still returns the closest available passages; this phase only exposes the resulting signals.

In [ ]:
query_scenarios = [
    ('Easy / obvious', 'Where is the Mona Lisa displayed?'),
    ('Paraphrased', 'Which orbiting observatory takes its name from an astronomer?'),
    ('Difficult / ambiguous', 'What happened in 1991?'),
    ('Answer absent from corpus', 'What is the capital of Canada?'),
]

for scenario, query in query_scenarios:
    print(f'\nSCENARIO: {scenario}')
    inspect_query(query)